#### Análise Exploratória de Dados: E-commerce Brasileiro (Olist)

**Objetivo:** Este notebook apresenta uma análise exploratória do conjunto de dados público da Olist, a maior loja de departamentos em marketplaces brasileiros. O foco é extrair insights acionáveis sobre logística, comportamento de pagamento, desempenho de produtos e satisfação do cliente para responder a perguntas estratégicas de negócio.

**Autores:** Kaike Brito Leitão, Enrico Santos Navajas e Mario

```mermaid
erDiagram
    CUSTOMERS {
        string customer_id PK
        string customer_unique_id
        string customer_zip_code_prefix FK
        string customer_city
        string customer_state
    }
    
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        datetime order_purchase_timestamp
        datetime order_approved_at
        datetime order_delivered_carrier_date
        datetime order_delivered_customer_date
        datetime order_estimated_delivery_date
    }
    
    ORDER_ITEMS {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        datetime shipping_limit_date
        float price
        float freight_value
    }
    
    PRODUCTS {
        string product_id PK
        string product_category_name FK
        int product_name_lenght
        int product_description_lenght
        int product_photos_qty
        float product_weight_g
        float product_length_cm
        float product_height_cm
        float product_width_cm
    }
    
    SELLERS {
        string seller_id PK
        string seller_zip_code_prefix FK
        string seller_city
        string seller_state
    }
    
    ORDER_PAYMENTS {
        string order_id PK, FK
        int payment_sequential PK
        string payment_type
        int payment_installments
        float payment_value
    }
    
    ORDER_REVIEWS {
        string review_id PK
        string order_id FK
        int review_score
        string review_comment_title
        string review_comment_message
        datetime review_creation_date
        datetime review_answer_timestamp
    }
    
    GEOLOCATION {
        string geolocation_zip_code_prefix PK
        float geolocation_lat
        float geolocation_lng
        string geolocation_city
        string geolocation_state
    }
    
    TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }

    %% Relacionamentos mapeados para a regra de negócio %%
    CUSTOMERS ||--o{ ORDERS : "realiza_pedido"
    ORDERS ||--|{ ORDER_ITEMS : "contem_itens"
    ORDERS ||--|{ ORDER_PAYMENTS : "possui_pagamentos"
    ORDERS ||--o{ ORDER_REVIEWS : "recebe_avaliacao"
    PRODUCTS ||--o{ ORDER_ITEMS : "compoe"
    SELLERS ||--o{ ORDER_ITEMS : "fornece"
    GEOLOCATION ||--o{ CUSTOMERS : "localiza_cliente"
    GEOLOCATION ||--o{ SELLERS : "localiza_vendedor"
    TRANSLATION ||--o{ PRODUCTS : "traduz_categoria"


In [7]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import logging
from pathlib import Path
from typing import Dict, Tuple, List
from IPython.display import display
from IPython.display import display
 
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [8]:
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

In [9]:
# ── Constantes ────────────────────────────────────────────────────────────────
RAW          = Path("../dataframes/raw/")           # ajustar para o seu path local
OUT          = Path("../dataframes/processed/")
OUT.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE    = 0.20
OUTLIER_P99  = 46    # dias — P99 calculado nos dados reais
PALETA       = "#2563EB"
 
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

In [10]:
# =============================================================================
# SEÇÃO 1 — CARREGAMENTO DAS TABELAS
# =============================================================================
 
def carregar_tabelas(path: Path) -> Dict[str, pd.DataFrame]:
    """
    Carrega os 9 CSVs do Olist em um dicionário tipado.
    
    Args:
        path: Caminho da pasta contendo os CSVs.
    Returns:
        Dicionário {nome_tabela: DataFrame}
    """
    arquivos = {
        "orders":      "olist_orders_dataset.csv",
        "order_items": "olist_order_items_dataset.csv",
        "payments":    "olist_order_payments_dataset.csv",
        "products":    "olist_products_dataset.csv",
        "sellers":     "olist_sellers_dataset.csv",
        "customers":   "olist_customers_dataset.csv",
        "geolocation": "olist_geolocation_dataset.csv",
        "translation": "product_category_name_translation.csv",
    }
    dfs = {}
    for nome, arq in arquivos.items():
        dfs[nome] = pd.read_csv(path / arq)
        logger.info(f"✅ {nome}: {dfs[nome].shape}")
    return dfs
 
dfs = carregar_tabelas(RAW)

INFO | ✅ orders: (99441, 8)
INFO | ✅ order_items: (112650, 7)
INFO | ✅ payments: (103886, 5)
INFO | ✅ products: (32951, 9)
INFO | ✅ sellers: (3095, 4)
INFO | ✅ customers: (99441, 5)
INFO | ✅ geolocation: (1000163, 5)
INFO | ✅ translation: (71, 2)


In [ ]:
# =============================================================================
# SEÇÃO 2 — ENGENHARIA DE FEATURES & CONSTRUÇÃO DO DATASET
# =============================================================================
# Cada linha do dataset final = 1 pedido entregue
# Tabelas utilizadas: orders (hub) + order_items + payments + products +
#                     sellers + customers + geolocation + translation
 
def construir_dataset(dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Integra as 8 tabelas em um único DataFrame para modelagem.
 
    Features geradas por domínio:
    ┌─ Temporais   ─┐  estimativa_prazo, dia_semana_compra, hora_compra, mes_compra
    ├─ Logísticas  ─┤  dias_ate_aprova_h, n_items, n_sellers
    ├─ Financeiras ─┤  price_total, freight_total, payment_value_total, installments
    ├─ Produto     ─┤  product_weight_g, volume_cm3, product_photos_qty, categoria
    └─ Geográficas ─┘  customer_state, seller_state, geolocation_lat, geolocation_lng
 
    Returns:
        DataFrame bruto (antes da limpeza de outliers).
    """
    orders = dfs["orders"].copy()
 
    # ── Conversão de datas ────────────────────────────────────────────────────
    for col in ["order_purchase_timestamp", "order_delivered_customer_date",
                "order_estimated_delivery_date", "order_approved_at",
                "order_delivered_carrier_date"]:
        orders[col] = pd.to_datetime(orders[col])
 
    # ── Filtrar apenas pedidos entregues ──────────────────────────────────────
    df = orders[orders["order_status"] == "delivered"].copy()
    df = df.dropna(subset=["order_delivered_customer_date"])
 
    # ── TARGET: dias totais entre compra e entrega ────────────────────────────
    df["dias_entrega"] = (
        df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
    ).dt.days
 
    # ── Features temporais ────────────────────────────────────────────────────
    df["estimativa_prazo"]   = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days # dias entre compra e prazo estimado
    df["dia_semana_compra"]  = df["order_purchase_timestamp"].dt.dayofweek   # 0=Seg, 6=Dom
    df["hora_compra"]        = df["order_purchase_timestamp"].dt.hour # horário da compra (0-23)
    df["mes_compra"]         = df["order_purchase_timestamp"].dt.month # mês da compra (1-12)
    df["dias_ate_aprova_h"]  = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600 # horas entre compra e aprovação do pedido
 
    df = df[["order_id", "customer_id", "dias_entrega", "estimativa_prazo",
             "dia_semana_compra", "hora_compra", "mes_compra", "dias_ate_aprova_h"]]
 
    # ── Agregação de Order Items (1 linha por pedido) ─────────────────────────
    items = dfs["order_items"]
    items_agg = items.groupby("order_id").agg(
        n_items        = ("order_item_id",  "count"), # número de itens no pedido
        price_total    = ("price",          "sum"), # valor total dos produtos (sem frete)
        freight_total  = ("freight_value",  "sum"), # valor total do frete
        n_sellers      = ("seller_id",      "nunique"), # número de vendedores distintos no pedido
        product_id_1st = ("product_id",     "first"),   # produto principal do pedido
        seller_id_1st  = ("seller_id",      "first"), # vendedor do produto principal do pedido
    ).reset_index()
    df = df.merge(items_agg, on="order_id", how="left")
 
    # ── Agregação de Pagamentos ───────────────────────────────────────────────
    pay = dfs["payments"]
    pay_agg = pay.groupby("order_id").agg(
        payment_value_total      = ("payment_value",       "sum"), # valor total pago (pode ser diferente de price_total + freight_total por causa de descontos, cupons, etc)
        payment_installments_max = ("payment_installments", "max"), # número máximo de parcelas (pode indicar compras mais caras)
    ).reset_index()
    # Tipo de pagamento principal (1ª transação)
    pay_type = (pay.sort_values("payment_sequential")
                   .groupby("order_id")["payment_type"].first().reset_index())
    pay_agg = pay_agg.merge(pay_type, on="order_id")
    df = df.merge(pay_agg, on="order_id", how="left")
 
    # ── Produtos + Tradução de Categoria ─────────────────────────────────────
    prod = dfs["products"].merge(dfs["translation"], on="product_category_name", how="left")
    prod["volume_cm3"] = (
        prod["product_length_cm"] * prod["product_height_cm"] * prod["product_width_cm"]
    )
    df = df.merge(
        prod[["product_id", "product_category_name_english",
              "product_weight_g", "volume_cm3", "product_photos_qty"]],
        left_on="product_id_1st", right_on="product_id", how="left"
    )
 
    # ── Vendedores ────────────────────────────────────────────────────────────
    df = df.merge(
        dfs["sellers"][["seller_id", "seller_state"]],
        left_on="seller_id_1st", right_on="seller_id", how="left"
    )
 
    # ── Clientes ──────────────────────────────────────────────────────────────
    df = df.merge(
        dfs["customers"][["customer_id", "customer_state", "customer_zip_code_prefix"]],
        on="customer_id", how="left"
    )
 
    # ── Geolocalização do cliente (mediana por ZIP para reduzir ruído) ────────
    geo = dfs["geolocation"]
    geo_med = (geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
                  .median().reset_index())
    cust_geo = (dfs["customers"][["customer_id", "customer_zip_code_prefix"]]
                .merge(geo_med, left_on="customer_zip_code_prefix",
                       right_on="geolocation_zip_code_prefix", how="left"))
    df = df.merge(cust_geo[["customer_id", "geolocation_lat", "geolocation_lng"]],
                  on="customer_id", how="left")
 
    # ── Remover colunas auxiliares ────────────────────────────────────────────
    df = df.drop(columns=[
        "product_id", "seller_id", "product_id_1st", "seller_id_1st",
        "customer_zip_code_prefix", "customer_id", "order_id"
    ], errors="ignore")
 
    logger.info(f"Dataset bruto: {df.shape} | Target nulos: {df['dias_entrega'].isna().sum()}")
    return df
 

def exibir_tabela(df: pd.DataFrame, titulo: str = None) -> None:
    if titulo:
        print(f"\n📋 {titulo}")
    display(df)
 
df_raw = construir_dataset(dfs)
 
feature_summary = pd.DataFrame([
    {"Domínio": "Temporais", "Features": "estimativa_prazo, dia_semana_compra, hora_compra, mes_compra"},
    {"Domínio": "Logísticas", "Features": "dias_ate_aprova_h, n_items, n_sellers"},
    {"Domínio": "Financeiras", "Features": "price_total, freight_total, payment_value_total, payment_installments_max"},
    {"Domínio": "Produto", "Features": "product_weight_g, volume_cm3, product_photos_qty, product_category_name_english"},
    {"Domínio": "Geográficas", "Features": "customer_state, seller_state, geolocation_lat, geolocation_lng"},
    {"Domínio": "Pagamento", "Features": "payment_type"},
])
print("\n📌 Sumário de features geradas")
exibir_tabela(feature_summary)


📌 Sumário de features geradas


,Domínio,Features
0,Temporais,"estimativa_prazo, dia_semana_compra, hora_comp..."
1,Geográficas,"dist_km, delta_lat, delta_lng, mesma_uf, mesma..."
2,Logísticas,"dias_ate_aprova_h, n_items, n_sellers, freight..."
3,Seller,"seller_avg_delivery, seller_std_delivery, sell..."
4,Produto,"product_weight_g, volume_cm3, densidade_g_cm3,..."
5,Pagamento,"payment_type, payment_value_total, payment_ins..."


In [ ]:
# =============================================================================
# SEÇÃO 3 — FILTRAGEM E LIMPEZA DOS DADOS
# =============================================================================
 
def limpar_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove ruídos, inconsistências e outliers extremos.
 
    Regras aplicadas:
    1. Remove linhas com target nulo (8 linhas — dados incompletos de sistema)
    2. Remove entregas de 0 dias (impossível operacionalmente — erro de timestamp)
    3. Remove outliers acima do P99 (46 dias): eventos excepcionais que distorceriam
       os modelos. P99 calculado empiricamente nos dados reais.
    4. Remove estimativa_prazo nula ou não positiva (dados corrompidos)
 
    Returns:
        DataFrame limpo, reindexado.
    """
    n_inicial = len(df)
 
    df = df.dropna(subset=["dias_entrega"])           # 8 linhas com target nulo
    df = df[df["dias_entrega"] > 0]                   # remove 0 dias (erro de data)
    df = df[df["dias_entrega"] <= OUTLIER_P99]        # remove outliers acima do P99
    df = df.dropna(subset=["estimativa_prazo"])
    df = df[df["estimativa_prazo"] > 0]
 
    n_final = len(df)
    logger.info(f"Limpeza: {n_inicial:,} → {n_final:,} linhas ({n_inicial-n_final:,} removidas)")
    return df.reset_index(drop=True)
 
df_clean = limpar_dataset(df_raw)
 
print(f"✅ Dataset limpo: {df_clean.shape}")
target_stats = df_clean["dias_entrega"].describe().round(2).to_frame().T
target_stats["skewness"] = df_clean["dias_entrega"].skew().round(3)
target_stats.index = ["dias_entrega"]
print("\n📊 Estatísticas do target:")
exibir_tabela(target_stats)
 

INFO | Limpeza: 96,470 → 95,577 linhas (893 removidas)


✅ Dataset limpo: (95577, 39)

📊 Estatísticas do target:


,count,mean,std,min,25%,50%,75%,max,skewness
dias_entrega,95577.0,11.62,7.81,1.0,6.0,10.0,15.0,46.0,1.429


In [ ]:
# =============================================================================
# DOWNLOAD DO DATASET LIMPO
# =============================================================================
# Exporta o dataset final (após feature engineering + limpeza) para CSV,
# permitindo reuso em outros notebooks sem re-executar todo o pipeline.
# O arquivo inclui target + todas as 38 features enriquecidas.
 
_EXPORT_PATH = OUT / "olist_dataset_enriquecido.csv"
df_clean.to_csv(_EXPORT_PATH, index=False)
print(f"\n💾 Dataset exportado → {_EXPORT_PATH}")
print(f"   Shape: {df_clean.shape[0]:,} linhas × {df_clean.shape[1]} colunas")
print(f"   Tamanho: {_EXPORT_PATH.stat().st_size / 1_048_576:.1f} MB")
print(f"\n   Para carregar em outro notebook:")
print(f"   df_clean = pd.read_csv('{_EXPORT_PATH}')")

In [ ]:
# =============================================================================
# SEÇÃO 4 — EDA COMPLETA (dataset enriquecido)
# =============================================================================
# 9 análises cobrindo: target, correlações, outliers, geografia, distância,
# geolocalização, histórico do vendedor, variáveis financeiras e produto.
 
# ── 4.1 Distribuição do Target ────────────────────────────────────────────────
# Pergunta: como se distribui o tempo de entrega?
# O target apresenta assimetria positiva (skewness = 1.43): a maioria das
# entregas chega em 10 dias, mas existe uma cauda de pedidos mais lentos.
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
axes[0].hist(df_clean["dias_entrega"], bins=46, color=PALETA, edgecolor="white", alpha=0.9)
axes[0].axvline(df_clean["dias_entrega"].mean(),   color="red",    ls="--", lw=1.5,
                label=f"Média {df_clean['dias_entrega'].mean():.1f}d")
axes[0].axvline(df_clean["dias_entrega"].median(), color="orange", ls="--", lw=1.5,
                label=f"Mediana {df_clean['dias_entrega'].median():.0f}d")
axes[0].set_title("Distribuição de Dias de Entrega (Variável Alvo)")
axes[0].set_xlabel("Dias de Entrega")
axes[0].set_ylabel("Frequência")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[0].legend()
 
meses = sorted(df_clean["mes_compra"].unique())
data_box = [df_clean[df_clean["mes_compra"] == m]["dias_entrega"].values for m in meses]
axes[1].boxplot(data_box, positions=meses, widths=0.6, patch_artist=True, showfliers=False,
                boxprops=dict(facecolor=PALETA, alpha=0.7),
                medianprops=dict(color="white", linewidth=2))
axes[1].set_title("Dias de Entrega por Mês de Compra")
axes[1].set_xlabel("Mês")
axes[1].set_ylabel("Dias de Entrega")
axes[1].set_xticks(meses)
axes[1].set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun",
                          "Jul","Ago","Set","Out","Nov","Dez"], fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "fig1_target_distribuicao.png", bbox_inches="tight")
plt.show()
 
 
# ── 4.2 Correlação com o target (barras horizontais — todas as features) ──────
# Pergunta: quais features mais explicam o tempo de entrega?
# Gráfico de barras colorido por direção (positivo=azul / negativo=vermelho).
# Leitura imediata da força e direção de cada feature.
 
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr_all = (df_clean[num_cols].corr()["dias_entrega"]
              .drop("dias_entrega")
              .sort_values(key=abs, ascending=False)
              .head(20))
 
fig, ax = plt.subplots(figsize=(10, 8))
cores_bar = ["#2563EB" if v > 0 else "#DC2626" for v in corr_all.values]
bars = ax.barh(corr_all.index[::-1], corr_all.values[::-1],
               color=cores_bar[::-1], edgecolor="white", alpha=0.85)
ax.axvline(0, color="gray", lw=0.8)
for bar, val in zip(bars, corr_all.values[::-1]):
    ax.text(val + (0.005 if val >= 0 else -0.005),
            bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center",
            ha="left" if val >= 0 else "right", fontsize=9)
ax.set_title("Correlação de Pearson — Features × dias_entrega (top 20)")
ax.set_xlabel("r de Pearson")
ax.set_xlim(-0.55, 0.62)
plt.tight_layout()
plt.savefig(OUT / "fig2_correlacoes.png", bbox_inches="tight")
plt.show()
 
# Tabela impressa das top 10
corr_target_table = (corr_all.head(10)
                     .round(3)
                     .reset_index()
                     .rename(columns={"index": "feature", "dias_entrega": "pearson_r"}))
print("\n🎯 Top 10 correlações com dias_entrega:")
exibir_tabela(corr_target_table)
 
 
# ── 4.3 Matriz de Correlação (heatmap) — features selecionadas ───────────────
# Pergunta: existe multicolinearidade entre features?
# Restringe às 15 features mais relevantes para legibilidade.
# Detecta grupos de features altamente correlacionadas entre si (ex:
# dist_km × media_dias_uf_cliente: r=0.62 → monitorar multicolinearidade).
 
key_feats = [
    "dias_entrega", "dist_km", "media_dias_uf_cliente", "estimativa_prazo",
    "mesma_uf", "seller_avg_delivery", "mesma_regiao", "geolocation_lat",
    "freight_total", "delta_lat", "seller_std_delivery",
    "dias_ate_aprova_h", "freight_ratio", "product_weight_g", "volume_cm3",
]
corr_mat = df_clean[key_feats].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
 
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.4, annot_kws={"size": 8}, ax=ax)
ax.set_title("Matriz de Correlação — Features Selecionadas", pad=15)
plt.tight_layout()
plt.savefig(OUT / "fig3_correlacao_heatmap.png", bbox_inches="tight")
plt.show()
 
 
# ── 4.4 Detecção de Outliers (IQR) ────────────────────────────────────────────
# Pergunta: quais features têm distribuições com caudas extremas?
# Todas as novas features geográficas e de seller também são inspecionadas.
# Outliers NÃO são removidos das features — apenas identificados para informar
# a escolha do scaler (StandardScaler vs. RobustScaler).
 
features_box = [
    "price_total", "freight_total", "product_weight_g",
    "volume_cm3",  "payment_value_total", "dias_ate_aprova_h",
    "n_items",     "dist_km",             "seller_avg_delivery",
]
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
 
for i, col in enumerate(features_box):
    dados = df_clean[col].dropna()
    q1, q3 = dados.quantile(0.25), dados.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((dados < q1 - 1.5 * iqr) | (dados > q3 + 1.5 * iqr)).sum()
    axes[i].boxplot(dados, vert=True, showfliers=True,
                    flierprops=dict(marker=".", markersize=2, alpha=0.3, color="red"))
    axes[i].set_title(f"{col}\n({n_outliers:,} outliers IQR)", fontsize=9)
 
plt.suptitle("Detecção de Outliers — Boxplot (IQR)", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "fig4_outliers.png", bbox_inches="tight")
plt.show()
 
 
# ── 4.5 Análise Geográfica — UF + Mapa ────────────────────────────────────────
# Pergunta: a localização do cliente afeta o prazo?
# Barras horizontais por UF com rótulo numérico + scatter lat/lng colorido
# pelo target. Destaque em vermelho para o estado mais lento (AM: 20,7d)
# e verde para o mais rápido (SP: 8,3d).
 
estado_media = df_clean.groupby("customer_state")["dias_entrega"].mean().sort_values(ascending=True)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
cores_uf = ["#DC2626" if e == estado_media.idxmax()
            else ("#16A34A" if e == estado_media.idxmin() else PALETA)
            for e in estado_media.index]
axes[0].barh(estado_media.index, estado_media.values, color=cores_uf, edgecolor="white")
axes[0].axvline(df_clean["dias_entrega"].mean(), color="gray", ls="--", lw=1, label="Média geral")
for i, (uf, val) in enumerate(zip(estado_media.index, estado_media.values)):
    axes[0].text(val + 0.1, i, f"{val:.1f}d", va="center", fontsize=7)
axes[0].set_title("Tempo Médio de Entrega por Estado (UF) do Cliente")
axes[0].set_xlabel("Dias")
axes[0].legend(fontsize=9)
 
df_geo = df_clean.dropna(subset=["geolocation_lat", "geolocation_lng"])
sc = axes[1].scatter(
    df_geo["geolocation_lng"], df_geo["geolocation_lat"],
    c=df_geo["dias_entrega"], cmap="RdYlGn_r",
    alpha=0.15, s=2, vmin=0, vmax=40,
)
plt.colorbar(sc, ax=axes[1], label="Dias de Entrega")
axes[1].set_xlim(-75, -34)
axes[1].set_ylim(-34, 6)
axes[1].set_title("Mapa Geográfico: Dias de Entrega por Localização do Cliente")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
 
plt.tight_layout()
plt.savefig(OUT / "fig5_geografico.png", bbox_inches="tight")
plt.show()
 
print(f"\n🗺️ Estado mais lento:  {estado_media.idxmax()} — {estado_media.max():.1f} dias")
print(f"🗺️ Estado mais rápido: {estado_media.idxmin()} — {estado_media.min():.1f} dias")
 
 
# ── 4.6 NOVA — Distância Haversine por Faixa + Dispersão ─────────────────────
# Pergunta: a distância real entre cliente e vendedor explica o prazo?
# Painel duplo: barras por faixa operacional (0–100km até 2000km+) com rótulo
# de dias médios + scatter com linha de tendência (r=0.44).
# Insight: entregas locais (<100km) chegam em 5,9d; entregas extremas em 19,3d.
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
faixa_labels = [
    "0–100km\n(local)", "100–300km\n(regional)", "300–600km\n(inter-reg)",
    "600–1000km\n(longa)", "1000–2000km\n(muito longa)", "2000km+\n(extrema)",
]
faixa_stats = (df_clean.groupby("faixa_dist_km")["dias_entrega"]
                .agg(["mean", "count"]).reset_index())
cores_faixa = ["#16A34A", "#4ADE80", "#FBBF24", "#F97316", "#EF4444", "#991B1B"]
 
bars = axes[0].bar(range(len(faixa_labels)), faixa_stats["mean"].values,
                   color=cores_faixa, edgecolor="white", alpha=0.9)
axes[0].set_xticks(range(len(faixa_labels)))
axes[0].set_xticklabels(faixa_labels, fontsize=8)
axes[0].set_title("Média de Entrega por Faixa de Distância (Haversine)")
axes[0].set_ylabel("Dias")
axes[0].axhline(df_clean["dias_entrega"].mean(), color="gray", ls="--", lw=1, label="Média geral")
axes[0].legend()
for bar, val in zip(bars, faixa_stats["mean"].values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.2, f"{val:.1f}d",
                 ha="center", fontsize=9, fontweight="bold")
 
df_scatter = df_clean.dropna(subset=["dist_km"]).sample(10_000, random_state=42)
axes[1].scatter(df_scatter["dist_km"], df_scatter["dias_entrega"],
                alpha=0.08, s=4, color=PALETA)
z = np.polyfit(df_scatter["dist_km"], df_scatter["dias_entrega"], 1)
x_line = np.linspace(0, 3_800, 200)
axes[1].plot(x_line, np.poly1d(z)(x_line), color="red", lw=2, label="Tendência (r=0.44)")
axes[1].set_xlabel("Distância cliente↔vendedor (km)")
axes[1].set_ylabel("Dias de Entrega")
axes[1].set_title("Dispersão: dist_km × dias_entrega")
axes[1].legend()
 
plt.tight_layout()
plt.savefig(OUT / "fig6_distancia.png", bbox_inches="tight")
plt.show()
 
 
# ── 4.7 NOVA — Mesma UF + Macrorregião + Histórico do Vendedor ───────────────
# Pergunta: a relação cliente-vendedor e o histórico do seller explicam o prazo?
# Painel triplo:
#   • Violin plot: mesma_uf=0 (14,1d) vs mesma_uf=1 (7,3d) — gap de ~93%
#   • Boxplot por macrorregião: Norte > Nordeste >> Sul > Sudeste
#   • Scatter: seller_avg_delivery histórico × prazo real (r=0.35)
 
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
 
# Violin: mesma UF
for idx, (v, lbl) in enumerate([(0, "UF diferente\n14,1d"), (1, "Mesma UF\n7,3d")]):
    dados = df_clean[df_clean["mesma_uf"] == v]["dias_entrega"]
    vp = axes[0].violinplot(dados, positions=[idx], showmedians=True, showextrema=False, widths=0.6)
    for pc in vp["bodies"]:
        pc.set_facecolor("#EF4444" if v == 0 else "#16A34A")
        pc.set_alpha(0.7)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(["UF diferente\n14,1d", "Mesma UF\n7,3d"])
axes[0].set_title("Dias de Entrega: Mesma UF vs. UF Diferente")
axes[0].set_ylabel("Dias")
 
# Boxplot por macrorregião
regioes = (df_clean.groupby("regiao_cliente")["dias_entrega"]
           .median().sort_values(ascending=False).index.tolist())
data_reg = [df_clean[df_clean["regiao_cliente"] == r]["dias_entrega"].values for r in regioes]
bp = axes[1].boxplot(data_reg, labels=regioes, patch_artist=True,
                     showfliers=False, medianprops=dict(color="white", lw=2))
cores_reg = ["#DC2626", "#EF4444", "#FBBF24", "#16A34A", "#2563EB"]
for patch, cor in zip(bp["boxes"], cores_reg):
    patch.set_facecolor(cor)
    patch.set_alpha(0.7)
axes[1].set_title("Distribuição por Macrorregião do Cliente")
axes[1].set_ylabel("Dias")
plt.setp(axes[1].xaxis.get_majorticklabels(), fontsize=8)
 
# Scatter: histórico do vendedor
df_sell = df_clean.dropna(subset=["seller_avg_delivery"]).sample(8_000, random_state=42)
axes[2].scatter(df_sell["seller_avg_delivery"].clip(upper=40), df_sell["dias_entrega"],
                alpha=0.08, s=4, color="#7C3AED")
z2 = np.polyfit(df_sell["seller_avg_delivery"].clip(upper=40), df_sell["dias_entrega"], 1)
xl = np.linspace(0, 40, 100)
axes[2].plot(xl, np.poly1d(z2)(xl), color="red", lw=2, label="r = 0.35")
axes[2].set_xlabel("seller_avg_delivery (dias)")
axes[2].set_ylabel("Dias de Entrega")
axes[2].set_title("Histórico do Vendedor × Prazo Real")
axes[2].legend()
 
plt.tight_layout()
plt.savefig(OUT / "fig7_geo_seller.png", bbox_inches="tight")
plt.show()
 
# Tabela: média por UF com ranking
uf_rank = (df_clean.groupby("customer_state")["dias_entrega"]
           .agg(["mean", "median", "count"])
           .round(1)
           .sort_values("mean", ascending=False)
           .rename(columns={"mean": "media_dias", "median": "mediana_dias", "count": "n_pedidos"})
           .reset_index())
print("\n🗺️ Ranking de prazo médio por UF:")
exibir_tabela(uf_rank)
 
 
# ── 4.8 NOVA — Variáveis Financeiras ─────────────────────────────────────────
# Pergunta: o comportamento financeiro do pedido afeta o prazo?
# Três análises:
#   • freight_ratio (frete/preço) por faixas → ratio alto = distante/pesado
#   • Tipo de pagamento: boleto (12,6d) > voucher > credit_card > debit_card
#   • Parcelamento: mais parcelas = ticket maior = produto mais pesado?
 
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
 
# Freight ratio
df_clean["freight_ratio_bin"] = pd.cut(df_clean["freight_ratio"].clip(upper=2.0), bins=8)
fr_stats = df_clean.groupby("freight_ratio_bin", observed=True)["dias_entrega"].mean()
axes[0].bar(range(len(fr_stats)), fr_stats.values, color=PALETA, edgecolor="white", alpha=0.85)
axes[0].set_xticks(range(len(fr_stats)))
axes[0].set_xticklabels([str(i) for i in fr_stats.index], rotation=45, fontsize=7)
axes[0].set_title("Frete Relativo (freight_ratio) × Prazo Médio")
axes[0].set_ylabel("Média de Dias")
axes[0].grid(axis="y", alpha=0.3)
 
# Payment type
pt_stats = df_clean.groupby("payment_type")["dias_entrega"].mean().sort_values(ascending=False)
cores_pt = ["#EF4444", "#F97316", "#16A34A", "#2563EB"][:len(pt_stats)]
bars_pt = axes[1].bar(pt_stats.index, pt_stats.values, color=cores_pt, edgecolor="white", alpha=0.9)
for bar, val in zip(bars_pt, pt_stats.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.1, f"{val:.1f}d",
                 ha="center", fontsize=10, fontweight="bold")
axes[1].set_title("Média de Entrega por Tipo de Pagamento")
axes[1].set_ylabel("Dias")
 
# Parcelamento
inst = (df_clean.groupby("payment_installments_max")["dias_entrega"]
        .mean().reset_index().head(12))
axes[2].plot(inst["payment_installments_max"], inst["dias_entrega"],
             marker="o", color=PALETA, lw=2, markersize=5)
axes[2].set_title("Parcelamento × Prazo Médio de Entrega")
axes[2].set_xlabel("Máximo de parcelas")
axes[2].set_ylabel("Dias")
axes[2].grid(axis="y", alpha=0.3)
 
plt.tight_layout()
plt.savefig(OUT / "fig8_financeiras.png", bbox_inches="tight")
plt.show()
 
 
# ── 4.9 NOVA — Produto: Categorias + Peso ─────────────────────────────────────
# Pergunta: o tipo e o peso do produto afetam o prazo?
# Top 15 categorias por mediana de entrega + histograma de prazo por faixa de peso.
# Insight: produtos pesados chegam mais tarde (peso > frete > prazo).
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
top_cats = df_clean["product_category_name_english"].value_counts().nlargest(15).index
cat_stats = (df_clean[df_clean["product_category_name_english"].isin(top_cats)]
             .groupby("product_category_name_english")
             .agg(mediana=("dias_entrega", "median"), contagem=("dias_entrega", "count"))
             .sort_values("mediana"))
 
axes[0].barh(cat_stats.index, cat_stats["mediana"], color=PALETA, edgecolor="white", alpha=0.85)
axes[0].axvline(df_clean["dias_entrega"].median(), color="red", ls="--", lw=1.5, label="Mediana geral")
for i, (idx, row) in enumerate(cat_stats.iterrows()):
    axes[0].text(row["mediana"] + 0.1, i, f"{row['mediana']:.0f}d", va="center", fontsize=8)
axes[0].set_title("Mediana de Entrega — Top 15 Categorias")
axes[0].set_xlabel("Mediana (dias)")
axes[0].legend(fontsize=9)
 
df_peso = df_clean.dropna(subset=["product_weight_g"]).copy()
df_peso["peso_bin"] = pd.cut(df_peso["product_weight_g"].clip(upper=10_000), bins=10)
pw = df_peso.groupby("peso_bin", observed=True)["dias_entrega"].mean()
axes[1].bar(range(len(pw)), pw.values, color=PALETA, edgecolor="white", alpha=0.85)
axes[1].set_xticks(range(len(pw)))
axes[1].set_xticklabels(
    [f"{int(b.left/1000)}–{int(b.right/1000)}kg" for b in pw.index],
    rotation=45, fontsize=8,
)
axes[1].set_title("Peso do Produto × Prazo Médio de Entrega")
axes[1].set_ylabel("Dias")
axes[1].grid(axis="y", alpha=0.3)
 
plt.tight_layout()
plt.savefig(OUT / "fig9_produto.png", bbox_inches="tight")
plt.show()
 


🎯 Correlações com dias_entrega (Pearson):


,feature,pearson_r
0,faixa_dist_km,0.471
1,media_dias_uf_cliente,0.461
2,dist_km,0.441
3,estimativa_prazo,0.430
4,mesma_uf,-0.413
5,seller_avg_delivery,0.349
6,mesma_regiao,-0.332
7,geolocation_lat,0.279
8,delta_lat,0.231
9,freight_total,0.185



🗺️ Estado mais lento:  AM — 24.8 dias
🗺️ Estado mais rápido: SP — 8.1 dias


In [ ]:
# =============================================================================
# SEÇÃO 5 — PRÉ-PROCESSAMENTO
# =============================================================================
 
# ── 5.1 Definição das features por tipo ──────────────────────────────────────
TARGET = "dias_entrega"
 
NUM_FEATURES: List[str] = [
    # ── Temporais ──────────────────────────────────────────────────────────────
    "estimativa_prazo",           # r=+0.43 — feature mais correlacionada
    "dias_ate_aprova_h",          # r=+0.10 — demora na aprovação indica problema
    "dia_semana_compra",          # padrão de dia da semana
    "hora_compra",                # padrão de hora do dia
    "mes_compra",                 # sazonalidade mensal
    "fim_de_semana",              # 1 se sáb/dom — atraso no processamento bancário
    "periodo_dia",                # 0=madrugada, 1=manhã, 2=tarde, 3=noite
    # ── Financeiras ────────────────────────────────────────────────────────────
    "price_total",                # valor total dos produtos
    "freight_total",              # r=+0.18 — frete alto = produto pesado/distante
    "payment_value_total",        # valor total pago (inclui juros)
    "payment_installments_max",   # parcelamento (proxy de valor do pedido)
    "freight_ratio",              # r=+0.10 — frete ÷ preço (proxy de distância)
    "avg_price_per_item",         # ticket médio por item
    "price_range",                # amplitude de preços no pedido
    # ── Itens ──────────────────────────────────────────────────────────────────
    "n_items",                    # quantidade de itens
    "n_sellers",                  # pedidos com múltiplos sellers = mais complexos
    # ── Produto ────────────────────────────────────────────────────────────────
    "product_weight_g",           # r=+0.08 — peso afeta frete e prazo
    "volume_cm3",                 # volume (L×A×C)
    "densidade_g_cm3",            # densidade g/cm³ — tipologia do produto
    "product_photos_qty",         # proxy de qualidade do anúncio
    # ── Geográficas numéricas ──────────────────────────────────────────────────
    "geolocation_lat",            # r=+0.28 — latitude do cliente
    "geolocation_lng",            # longitude do cliente
    "dist_km",                    # r=+0.44 — distância Haversine cliente↔vendedor
    "delta_lat",                  # r=+0.23 — componente Norte-Sul da rota
    "delta_lng",                  # r=+0.12 — componente Leste-Oeste da rota
    "faixa_dist_km",              # r=+0.47 — faixa operacional de distância
    "mesma_uf",                   # r=-0.41 — 1 se cliente e vendedor na mesma UF
    "mesma_regiao",               # r=-0.33 — 1 se mesma macrorregião
    "media_dias_uf_cliente",      # r=+0.46 — target encoding da UF destino
    # ── Seller ─────────────────────────────────────────────────────────────────
    "seller_avg_delivery",        # r=+0.35 — histórico médio do vendedor
    "seller_std_delivery",        # r=+0.15 — variabilidade histórica do vendedor
    "seller_n_orders",            # volume histórico de pedidos do vendedor
]
 
CAT_FEATURES: List[str] = [
    "payment_type",                    # credit_card / boleto / voucher / debit_card
    "customer_state",                  # 27 UFs (proxy de infraestrutura de destino)
    "seller_state",                    # UF do vendedor (proxy de infraestrutura origem)
    "regiao_cliente",                  # Norte / Nordeste / Centro-Oeste / Sudeste / Sul
    "regiao_vendedor",                 # macrorregião do vendedor
    "product_category_name_english",   # 71 categorias de produto
]
 
ALL_FEATURES = NUM_FEATURES + CAT_FEATURES
X = df_clean[ALL_FEATURES].copy()
y = df_clean[TARGET].copy()
 
print(f"\n📐 Features: {len(ALL_FEATURES)} total ({len(NUM_FEATURES)} numéricas + {len(CAT_FEATURES)} categóricas)")
print(f"📐 X shape: {X.shape} | y shape: {y.shape}")
 
# ── 5.2 Split treino/teste ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)
print(f"\n✂️  Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}")
 
# ── 5.3 Análise de nulos ──────────────────────────────────────────────────────
nulos = X_train.isnull().sum()
nulos_pct = (nulos[nulos > 0] / len(X_train) * 100).round(2)
nulos_df = nulos_pct.reset_index().rename(columns={"index": "feature", 0: "pct_nulos"})
nulos_df["tipo_nulo"] = nulos_df["feature"].map({
    "dias_ate_aprova_h":              "MAR",
    "product_category_name_english":  "MAR",
    "product_weight_g":               "MAR",
    "volume_cm3":                     "MAR",
    "densidade_g_cm3":                "MAR",
    "product_photos_qty":             "MAR",
    "geolocation_lat":                "MAR",
    "geolocation_lng":                "MAR",
    "dist_km":                        "MAR",
    "delta_lat":                      "MAR",
    "delta_lng":                      "MAR",
    "faixa_dist_km":                  "MAR",
}).fillna("MAR")
nulos_df["estrategia"] = nulos_df["feature"].map({
    "product_category_name_english": "SimpleImputer(most_frequent) → OrdinalEncoder",
    "regiao_cliente":                "SimpleImputer(most_frequent) → OrdinalEncoder",
    "regiao_vendedor":               "SimpleImputer(most_frequent) → OrdinalEncoder",
}).fillna("KNNImputer(k=5) → StandardScaler")
 
print("\n⚠️  Nulos no conjunto de treino:")
if nulos_df.empty:
    print("   Sem nulos.")
else:
    exibir_tabela(nulos_df)
 
# ── 5.4 ColumnTransformer ─────────────────────────────────────────────────────
#
# Intuição do ColumnTransformer: aplica transformações distintas em grupos de
# colunas em PARALELO, estimando parâmetros (mediana, categorias) APENAS no
# treino — prevenindo data leakage ao fit no teste.
#
# Pipeline NUMÉRICO:
#   KNNImputer(k=5)  → imputa pelo valor médio dos 5 vizinhos mais similares.
#                      Superior ao SimpleImputer(median) pois usa a estrutura
#                      multivariada dos dados (correlações entre features).
#   StandardScaler() → z-score (µ=0, σ=1). Necessário para Ridge (regularização
#                      L2 é sensível à escala). Neutro para árvores/boosting.
#
# Pipeline CATEGÓRICO:
#   SimpleImputer(most_frequent) → preenche nulos com a moda da categoria.
#   OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1):
#     → Mapeia cada categoria para um inteiro. Escolha vs. OneHotEncoder:
#       com 71 categorias em product_category, OHE geraria 71+ colunas esparsas.
#       OrdinalEncoder é eficiente para modelos de árvore (que não assumem
#       ordenação — eles splitam por limiar numérico, ignorando a ordem).
 
numeric_pipe = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler",  StandardScaler()),
])
 
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
 
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe,      NUM_FEATURES),
    ("cat", categorical_pipe,  CAT_FEATURES),
], remainder="drop")
 


📐 Features: 20 total (16 numéricas + 4 categóricas)
📐 X shape: (95577, 20) | y shape: (95577,)

✂️  Treino: 76,461 | Teste: 19,116

⚠️  Nulos no treino (% das linhas):
dias_ate_aprova_h                0.01
product_weight_g                 0.02
volume_cm3                       0.02
product_photos_qty               1.41
geolocation_lat                  0.28
geolocation_lng                  0.28
product_category_name_english    1.43


In [15]:
# =============================================================================
# SEÇÃO 6 — BENCHMARK DE MODELOS
# =============================================================================
#
# Modelos selecionados e justificativas:
#
# 1. Ridge (Regressão Linear + L2): baseline interpretável. Verifica se
#    o problema tem componente linear significativo. Coeficientes diretamente
#    interpretáveis. Regularização L2 lida com multicolinearidade entre
#    features de preço.
#
# 2. Decision Tree: baseline não-linear simples. Interpretável por regras.
#    Confirma se existem segmentações naturais nos dados (ex: "pedidos para
#    AM com peso > 5kg → prazo longo").
#
# 3. Random Forest: ensemble bagging — treina N árvores em subsets aleatórios
#    de dados E features, depois agrega por média. Reduz variância sem aumentar
#    viés. Robusto a outliers nas features.
#
# 4. Gradient Boosting: boosting sequencial — cada árvore corrige os resíduos
#    da anterior, minimizando a loss via gradiente descendente. Captura padrões
#    não-lineares complexos e interações entre features.
#
# 5. XGBoost: boosting otimizado com regularização L1/L2, shrinkage e column
#    subsampling. Estado da arte em dados tabulares estruturados. Lida com
#    missing values nativamente (aprende o melhor caminho para NaN).
 
def avaliar_regressao(nome: str, y_true, y_pred: np.ndarray) -> Dict:
    """
    Calcula RMSE, MAE, MAPE e R² para um modelo de regressão.
    
    Intuição das métricas:
    - RMSE: √MSE — penaliza erros grandes quadraticamente. Mesma unidade do target.
    - MAE:  erro médio absoluto — robusto a outliers. "Em média, erra X dias."
    - MAPE: erro percentual — interpretabilidade para o negócio.
    - R²:   proporção da variância explicada (1 = perfeito, 0 = baseline naive).
    """
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    mape = float(np.mean(np.abs((np.array(y_true) - y_pred) / np.clip(y_true, 1, None))) * 100)
    r2   = float(r2_score(y_true, y_pred))
    return {"modelo": nome, "RMSE": rmse, "MAE": mae, "MAPE_%": mape, "R2": r2}
 
 
MODELOS_PIPELINE: Dict[str, Pipeline] = {
    "Ridge": Pipeline([
        ("pre", preprocessor),
        ("reg", Ridge(alpha=10.0))
    ]),
    "DecisionTree": Pipeline([
        ("pre", preprocessor),
        ("reg", DecisionTreeRegressor(max_depth=8, min_samples_leaf=50, random_state=RANDOM_STATE))
    ]),
    "RandomForest": Pipeline([
        ("pre", preprocessor),
        ("reg", RandomForestRegressor(n_estimators=150, max_depth=12,
                                      min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    "GradientBoosting": Pipeline([
        ("pre", preprocessor),
        ("reg", GradientBoostingRegressor(n_estimators=150, learning_rate=0.08,
                                          max_depth=5, subsample=0.8,
                                          random_state=RANDOM_STATE))
    ]),
    "XGBoost": Pipeline([
        ("pre", preprocessor),
        ("reg", XGBRegressor(n_estimators=200, learning_rate=0.08, max_depth=6,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
    ]),
}
 
resultados = []
print("\n" + "="*65)
print("BENCHMARK — TREINAMENTO E AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*65)
 
for nome, pipe in MODELOS_PIPELINE.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    met = avaliar_regressao(nome, y_test, y_pred)
    resultados.append(met)
    print(f"  {nome:20s} | RMSE={met['RMSE']:.3f} | MAE={met['MAE']:.3f} | R²={met['R2']:.4f}")
 
resultado_df = pd.DataFrame(resultados).set_index("modelo").sort_values("RMSE")
print("\n📊 Ranking final")
exibir_tabela(resultado_df.round(4))
resultado_df.round(4).to_csv(OUT / "benchmark_resultados.csv")
 
 
# ── Gráfico de comparação ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
modelos_l = resultado_df.index.tolist()
rmses = resultado_df["RMSE"].values
r2s   = resultado_df["R2"].values
cores = ["#16a34a" if i == 0 else PALETA for i in range(len(modelos_l))]
 
axes[0].barh(modelos_l[::-1], rmses[::-1], color=cores[::-1], edgecolor="white", alpha=0.9)
for i, v in enumerate(rmses[::-1]):
    axes[0].text(v + 0.02, i, f"{v:.3f}", va="center", fontsize=9)
axes[0].set_title("RMSE — Teste (menor = melhor)"); axes[0].set_xlabel("RMSE (dias)")
 
axes[1].barh(modelos_l[::-1], r2s[::-1], color=cores[::-1], edgecolor="white", alpha=0.9)
for i, v in enumerate(r2s[::-1]):
    axes[1].text(v + 0.002, i, f"{v:.4f}", va="center", fontsize=9)
axes[1].set_title("R² — Teste (maior = melhor)"); axes[1].set_xlabel("R²"); axes[1].set_xlim(0, 1.05)
 
plt.suptitle("Comparação de Modelos de Regressão — Olist", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "fig6_benchmark.png", bbox_inches="tight")
plt.show()
 


BENCHMARK — TREINAMENTO E AVALIAÇÃO NO CONJUNTO DE TESTE
  Ridge                | RMSE=6.781 | MAE=4.904 | R²=0.2438
  DecisionTree         | RMSE=6.215 | MAE=4.457 | R²=0.3649
  RandomForest         | RMSE=6.017 | MAE=4.292 | R²=0.4047
  GradientBoosting     | RMSE=5.956 | MAE=4.238 | R²=0.4167
  XGBoost              | RMSE=5.875 | MAE=4.165 | R²=0.4324

📊 Ranking final


,RMSE,MAE,MAPE_%,R2
modelo,,,,
XGBoost,5.8752,4.1646,49.9114,0.4324
GradientBoosting,5.9557,4.2379,51.4331,0.4167
RandomForest,6.0168,4.2921,52.5662,0.4047
DecisionTree,6.2146,4.4566,55.2696,0.3649
Ridge,6.7814,4.9037,64.4398,0.2438


In [16]:
# =============================================================================
# SEÇÃO 7 — AJUSTE DE HIPERPARÂMETROS (RandomizedSearchCV — XGBoost)
# =============================================================================
#
# Intuição do RandomizedSearchCV:
# Amostra N combinações aleatórias do espaço de hiperparâmetros (Bergstra & Bengio,
# 2012). Para espaços grandes, é ~100x mais eficiente que GridSearchCV com
# desempenho similar. Com n_iter=20 e cv=5: 100 treinamentos totais.
#
# Hiperparâmetros e seus efeitos:
# • n_estimators:      mais árvores → menos variância, mais custo computacional
# • learning_rate:     shrinkage — taxa de contribuição de cada árvore (menor = mais robusto)
# • max_depth:         controla complexidade da árvore individual (maior = mais overfit)
# • subsample:         fração de linhas por árvore (bagging — reduz overfitting)
# • colsample_bytree:  fração de features por árvore (regulariza, análogo ao Random Forest)
# • reg_alpha (L1):    regularização Lasso — promove esparsidade de features
# • reg_lambda (L2):   regularização Ridge — suaviza pesos
 
print("\n" + "="*60)
print("AJUSTE DE HIPERPARÂMETROS — XGBoost (RandomizedSearchCV)")
print("="*60)
 
xgb_pipe_tunavel = Pipeline([
    ("pre", preprocessor),
    ("reg", XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
])
 
param_dist = {
    "reg__n_estimators":     [150, 200, 300, 400],
    "reg__learning_rate":    [0.03, 0.05, 0.08, 0.1, 0.15],
    "reg__max_depth":        [4, 5, 6, 7],
    "reg__subsample":        [0.7, 0.8, 0.9, 1.0],
    "reg__colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "reg__min_child_weight": [1, 3, 5, 10],
    "reg__reg_alpha":        [0, 0.01, 0.1, 1.0],
    "reg__reg_lambda":       [0.5, 1.0, 2.0, 5.0],
}
 
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
 
random_search = RandomizedSearchCV(
    estimator=xgb_pipe_tunavel,
    param_distributions=param_dist,
    n_iter=20,
    cv=kf,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
)
 
random_search.fit(X_train, y_train)
 
best_params_df = pd.DataFrame({
    "param": [p.replace("reg__", "") for p in random_search.best_params_.keys()],
    "value": list(random_search.best_params_.values())
})
print(f"\n✅ Melhores parâmetros encontrados:")
exibir_tabela(best_params_df)
print(f"\n   RMSE CV (melhor): {-random_search.best_score_:.4f} dias")
 
# Avaliação final no teste
y_pred_tunado = random_search.predict(X_test)
met_tunado = avaliar_regressao("XGBoost (tunado)", y_test, y_pred_tunado)

tunado_df = pd.DataFrame([{
    "modelo": "XGBoost (tunado)",
    "RMSE": met_tunado["RMSE"],
    "MAE": met_tunado["MAE"],
    "R2": met_tunado["R2"],
}]).set_index("modelo")
print(f"\n📊 XGBoost tunado no teste:")
exibir_tabela(tunado_df.round(4))


AJUSTE DE HIPERPARÂMETROS — XGBoost (RandomizedSearchCV)
Fitting 5 folds for each of 20 candidates, totalling 100 fits

✅ Melhores parâmetros encontrados:


,param,value
0,subsample,1.00
1,reg_lambda,1.00
2,reg_alpha,0.01
3,n_estimators,300.00
4,min_child_weight,5.00
5,max_depth,7.00
6,learning_rate,0.10
7,colsample_bytree,0.60



   RMSE CV (melhor): 5.9163 dias

📊 XGBoost tunado no teste:


,RMSE,MAE,R2
modelo,,,
XGBoost (tunado),5.82,4.1046,0.443


In [17]:
# =============================================================================
# SEÇÃO 8 — ANÁLISE DE RESÍDUOS E FEATURE IMPORTANCE
# =============================================================================
 
# ── 8.1 Resíduos ───────────────────────────────────────────────────────────────
y_pred_best = MODELOS_PIPELINE["XGBoost"].predict(X_test)
residuos = y_test.values - y_pred_best
 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
 
axes[0].scatter(y_pred_best, y_test.values, alpha=0.1, s=4, color=PALETA)
axes[0].plot([0, 46], [0, 46], "r--", lw=1.5, label="Predição Perfeita")
axes[0].set_xlabel("Predito (dias)"); axes[0].set_ylabel("Real (dias)")
axes[0].set_title("Real vs. Predito — XGBoost"); axes[0].legend()
 
axes[1].scatter(y_pred_best, residuos, alpha=0.1, s=4, color=PALETA)
axes[1].axhline(0, color="red", ls="--", lw=1.5)
axes[1].set_xlabel("Predito (dias)"); axes[1].set_ylabel("Resíduo (dias)")
axes[1].set_title("Resíduos vs. Predito")
 
axes[2].hist(residuos, bins=50, color=PALETA, edgecolor="white", alpha=0.9)
axes[2].axvline(0, color="red", ls="--", lw=1.5)
axes[2].set_xlabel("Resíduo (dias)"); axes[2].set_ylabel("Frequência")
axes[2].set_title(f"Distribuição de Resíduos | Bias={residuos.mean():.3f}d")
 
plt.tight_layout()
plt.savefig(OUT / "fig7_residuos.png", bbox_inches="tight")
plt.show()
 
# ── 8.2 Feature Importance ─────────────────────────────────────────────────────
fi = MODELOS_PIPELINE["XGBoost"].named_steps["reg"].feature_importances_
fi_df = (pd.DataFrame({"feature": ALL_FEATURES, "importance": fi})
           .sort_values("importance", ascending=True)
           .tail(15))
 
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df["feature"], fi_df["importance"], color=PALETA, edgecolor="white", alpha=0.85)
ax.set_title("Feature Importance — XGBoost (Top 15 por Gain médio)")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(OUT / "fig8_feature_importance.png", bbox_inches="tight")
plt.show()

In [18]:
# =============================================================================
# SEÇÃO 9 — RESUMO EXECUTIVO
# =============================================================================
 
best_modelo = resultado_df.iloc[0]

dataset_summary = pd.DataFrame([
    {"Métrica": "Pedidos entregues (após limpeza)", "Valor": f"{len(df_clean):,}"},
    {"Métrica": "Features numéricas", "Valor": len(NUM_FEATURES)},
    {"Métrica": "Features categóricas", "Valor": len(CAT_FEATURES)},
    {"Métrica": "Total de features", "Valor": len(ALL_FEATURES)},
    {"Métrica": "Target médio (dias)", "Valor": f"{df_clean['dias_entrega'].mean():.1f}"},
    {"Métrica": "Target mediana (dias)", "Valor": f"{df_clean['dias_entrega'].median():.0f}"},
    {"Métrica": "Target std (dias)", "Valor": f"{df_clean['dias_entrega'].std():.1f}"},
])
print("\n📌 Resumo do dataset")
exibir_tabela(dataset_summary)

final_model_summary = pd.DataFrame([
    {"Métrica": "Modelo final", "Valor": "XGBoost (tunado via RandomizedSearchCV)"},
    {"Métrica": "RMSE (teste)", "Valor": f"{met_tunado['RMSE']:.3f}"},
    {"Métrica": "MAE (teste)", "Valor": f"{met_tunado['MAE']:.3f}"},
    {"Métrica": "R² (teste)", "Valor": f"{met_tunado['R2']:.4f}"},
    {"Métrica": "Variância explicada", "Valor": f"{met_tunado['R2']*100:.1f}%"},
])
print("\n📌 Resumo executivo")
exibir_tabela(final_model_summary)

feature_impact = corr_target.head(5).reset_index().rename(columns={"index": "feature", "dias_entrega": "pearson_r"})
feature_impact["pearson_r"] = feature_impact["pearson_r"].round(3)
print("\n📌 Features mais impactantes")
exibir_tabela(feature_impact)

future_improvements = pd.DataFrame({
    "Melhoria": [
        "Distância Haversine (lat/lng cliente vs. vendedor)",
        "Histórico de desempenho do vendedor (% atrasos)",
        "Separação de modelos por macrorregião geográfica",
        "SHAP values para explicabilidade por previsão individual",
        "Stacking: XGBoost + LightGBM + RandomForest",
    ]
})
print("\n📌 Melhorias futuras")
exibir_tabela(future_improvements)

logger.info(f"✅ Pipeline concluído. Outputs salvos em {OUT}")
 


📌 Resumo do dataset


,Métrica,Valor
0,Pedidos entregues (após limpeza),"95,577"
1,Features numéricas,16
2,Features categóricas,4
3,Total de features,20
4,Target médio (dias),11.6
5,Target mediana (dias),10
6,Target std (dias),7.8



📌 Resumo executivo


,Métrica,Valor
0,Modelo final,XGBoost (tunado via RandomizedSearchCV)
1,RMSE (teste),5.820
2,MAE (teste),4.105
3,R² (teste),0.4430
4,Variância explicada,44.3%



📌 Features mais impactantes


,feature,pearson_r
0,faixa_dist_km,0.471
1,media_dias_uf_cliente,0.461
2,dist_km,0.441
3,estimativa_prazo,0.430
4,mesma_uf,-0.413



📌 Melhorias futuras


,Melhoria
0,Distância Haversine (lat/lng cliente vs. vende...
1,Histórico de desempenho do vendedor (% atrasos)
2,Separação de modelos por macrorregião geográfica
3,SHAP values para explicabilidade por previsão ...
4,Stacking: XGBoost + LightGBM + RandomForest


INFO | ✅ Pipeline concluído. Outputs salvos em ..\dataframes\processed
